# Day 15 — Project: CLI data tool
Build a small command-line script that ingests a CSV, cleans it, and outputs a summary with tests and linting.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-15`. Read
`python/ds-60day/companion-guides/day15_cli_project.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A command-line interface (CLI) has three layers: parse external strings,
call ordinary Python logic with typed values, then present a result and
choose an exit status. Keeping the core logic free from `sys.argv`,
printing, and process exit makes it reusable from tests and notebooks.

`argparse` defines flags, help, conversion, and validation close to the
command boundary. A `main(argv: list[str] | None = None) -> int`
function is testable because tests can pass an explicit list rather than
changing the real process arguments. The guarded entry point should do
little more than `raise SystemExit(main())`.

### Vocabulary

- **CLI:** a text interface driven by command-line arguments and exit status.
- **argument parser:** a component that converts command text into named values.
- **option:** a named flag such as `--limit`.
- **positional argument:** a value identified by its position.
- **exit status:** an integer process result where zero normally means success.
- **entry point:** the small boundary that starts application execution.

## Syntax anatomy

`parser.add_argument("--limit", type=int, default=10)` declares the
spelling, conversion, and fallback. `parser.parse_args(argv)` returns a
namespace of parsed values. Supplying `argv` makes tests deterministic;
`None` tells argparse to read the real process command line.

### Worked example 1 — Parse an explicit argument list

Exercise the CLI contract without touching the notebook process arguments. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import argparse

def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="rowtool")
    parser.add_argument("--limit", type=int, default=10)
    parser.add_argument("names", nargs="+")
    return parser

args = build_parser().parse_args(["--limit", "2", "Ada", "Lin", "Grace"])
(args.limit, args.names)

**Expected observation:** `(2, ['Ada', 'Lin', 'Grace'])`. Argparse converted `2` to an integer and collected positional names.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Keep core work independent of printing

A plain function can be tested and reused by the CLI. Predict first; then run the next cell.

In [ ]:
def select_names(names: list[str], *, limit: int) -> list[str]:
    if limit < 0:
        raise ValueError("limit must be non-negative")
    return [name.strip().title() for name in names[:limit]]

select_names(args.names, limit=args.limit)

**Expected observation:** `['Ada', 'Lin']`. Parsing and presentation remain outside the core transformation.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Print or test `vars(args)` when parsed values do not match the declared options.
2. Keep file I/O and transformation out of the parser-building function.
3. Return an exit code from `main`; avoid calling `sys.exit` deep inside reusable logic.
4. Test help, required arguments, invalid conversion, normal output, and output-file behavior separately.

**Alternative to compare:** A notebook suits interactive exploration; a CLI suits repeatable parameterized execution; a library function should hold shared core logic.

**Boundary to test:** Paths with spaces, missing files, invalid encodings, empty input, existing output files, and Windows shell quoting need tests.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

## Requirements
- Read CSV (pandas), drop NA, convert types.
- Output summary stats and save a cleaned CSV.
- Provide a CLI (`python tool.py --input data.csv --out out.csv`).
- Include pytest tests for core functions.
- Pass Ruff lint and format checks, mypy, and pytest.

## Hints
- Put core logic in functions so tests can import them.
- Use the standard-library `argparse` module for the CLI.
- Keep I/O separate from transforms.


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Build the Day 15 CLI with subcommands or flags that read a local CSV/JSON input, perform one documented transformation, and write or print a bounded result. **Architecture:** `build_parser()`, pure core function(s), and `main(argv=None) -> int`. **Constraints:** use `pathlib`, UTF-8, no notebook-only state, and no hard-coded absolute paths.
   **Verify:** run `--help`, one successful command, and one invalid invocation.

2. Add options for input path, output path, and a typed transformation parameter such as `--limit`.
   **Expected behavior:** argparse rejects invalid numeric text and the application returns a nonzero status for a missing input without a traceback aimed at beginners. **Constraint:** do not catch programming errors broadly.
   **Verify:** Exercise valid options, invalid integer text, and a missing input; assert parsed Python types and the documented nonzero exit status/message.

3. Write pytest tests that call core logic directly and call `main([...])` with temporary files. **Coverage:** happy path, empty input, missing path, invalid parameter, and output overwrite policy.
   **Verify:** assert return status, captured output, and exact file content without spawning a shell.

4. Package the CLI invocation behind `if __name__ == '__main__': raise SystemExit(main())`.
   **Expected behavior:** importing the module produces no output or process exit; `python -m ... --help` works from the documented package parent.
   **Verify:** test both import and module execution.

5. Create a short README usage block for Windows PowerShell and macOS/Linux showing repository-interpreter commands and an example with a path containing spaces. **Constraint:** do not mix Bash syntax into PowerShell.
   **Verify:** copy and run the command appropriate to your operating system.

### Additional mastery practice

Keep command parsing and file I/O at thin boundaries around pure, importable transformations. Make failures observable through exit codes.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

6. **Prediction:** Predict the Python types produced by `argparse` when `--input` uses `type=Path` and `--limit` uses `type=int`.
   **Progressive hint:** The parser performs declared conversions before `main` receives values.
   **Verify:** Call `parse_args` with explicit text and assert the parsed input is a `Path`, limit is an `int`, and defaults have the documented types.
7. **Tracing:** Trace one row through read → clean → summarize → write, and label which stages are I/O boundaries versus pure work.
   **Progressive hint:** A pure transform accepts and returns data without reading global state.
   **Verify:** For one fixture row, record the value/shape after every stage and assert only read/write touch files while the middle stages work from passed data.
8. **Implementation:** Add `--overwrite` and refuse to replace an existing output unless the flag is present.
   **Progressive hint:** Check the destination before performing the write.
   **Verify:** Use a temporary existing destination: assert refusal leaves content unchanged without the flag and `--overwrite` deliberately replaces it with the flag.
9. **Debugging:** Repair a module that parses arguments and writes files during import.
   **Progressive hint:** Move behavior into `main(argv)` and use the `__main__` guard.
   **Verify:** Import the module while capturing output/files and assert no parser or write occurs; then assert `main([...])` performs the intended operation.
10. **Edge case and explanation:** Define exit codes/messages for missing input, malformed data, existing output, and unexpected internal failure; decide which layers log.
   **Progressive hint:** Translate expected boundary failures once, without hiding tracebacks in tests.
   **Verify:** Exercise all four failure categories and assert their exit codes/messages; an injected unexpected error must remain visible in tests rather than being mislabeled.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.


## Run the project

Windows PowerShell:

```powershell
.\.venv\Scripts\python.exe tool.py --input data.csv --out artifacts\out.csv
.\.venv\Scripts\python.exe -m pytest
```

macOS/Linux:

```bash
.venv/bin/python tool.py --input data.csv --out artifacts/out.csv
.venv/bin/python -m pytest
```

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Build the Day 15 CLI with subcommands or flags that read a local CSV/JSON input, perform one documented transformation, and write or print a bounded result. **Architecture:** `build_parser()`, pure core function(s), and `main(argv=None) -> int`. **Constraints:** use `pathlib`, UTF-8, no notebook-only state, and no hard-coded absolute paths. **Verify:** run `--help`, one successful command, and one invalid invocation.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Build the Day 15 CLI with subcommands or flags that read a local CSV/JSON input, perform one documented transformation, and write or print a bounded result. `build_parser()`, pu...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add options for input path, output path, and a typed transformation parameter such as `--limit`. **Expected behavior:** argparse rejects invalid numeric text and the application returns a nonzero status for a missing input without a traceback aimed at beginners. **Constraint:** do not catch programming errors broadly. **Verify:** Exercise valid options, invalid integer text, and a missing input; assert parsed Python types and the documented nonzero exit status/message.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add options for input path, output path, and a typed transformation parameter such as `--limit`. argparse rejects invalid numeric text and the application returns a nonzero stat...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** Write pytest tests that call core logic directly and call `main([...])` with temporary files. **Coverage:** happy path, empty input, missing path, invalid parameter, and output overwrite policy. **Verify:** assert return status, captured output, and exact file content without spawning a shell.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Write pytest tests that call core logic directly and call `main([...])` with temporary files. happy path, empty input, missing path, invalid parameter, and output overwrite poli...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** Package the CLI invocation behind `if __name__ == '__main__': raise SystemExit(main())`. **Expected behavior:** importing the module produces no output or process exit; `python -m ... --help` works from the documented package parent. **Verify:** test both import and module execution.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Package the CLI invocation behind `if __name__ == '__main__': raise SystemExit(main())`. importing the module produces no output or process exit; `python -m ... --help` works fr...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** Create a short README usage block for Windows PowerShell and macOS/Linux showing repository-interpreter commands and an example with a path containing spaces. **Constraint:** do not mix Bash syntax into PowerShell. **Verify:** copy and run the command appropriate to your operating system.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Create a short README usage block for Windows PowerShell and macOS/Linux showing repository-interpreter commands and an example with a path containing spaces. do not mix Bash sy...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the Python types produced by `argparse` when `--input` uses `type=Path` and `--limit` uses `type=int`. **Progressive hint:** The parser performs declared conversions before `main` receives values. **Verify:** Call `parse_args` with explicit text and assert the parsed input is a `Path`, limit is an `int`, and defaults have the documented types.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Predict the Python types produced by `argparse` when `--input` uses `type=Path` and `--limit` uses `type=int`. The parser performs declared conversions before `main` receives va...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace one row through read → clean → summarize → write, and label which stages are I/O boundaries versus pure work. **Progressive hint:** A pure transform accepts and returns data without reading global state. **Verify:** For one fixture row, record the value/shape after every stage and assert only read/write touch files while the middle stages work from passed data.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Trace one row through read → clean → summarize → write, and label which stages are I/O boundaries versus pure work. A pure transform accepts and returns data without reading glo...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Add `--overwrite` and refuse to replace an existing output unless the flag is present. **Progressive hint:** Check the destination before performing the write. **Verify:** Use a temporary existing destination: assert refusal leaves content unchanged without the flag and `--overwrite` deliberately replaces it with the flag.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Add `--overwrite` and refuse to replace an existing output unless the flag is present. Check the destination before performing the write. Use a temporary existing destination: a...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 9 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a module that parses arguments and writes files during import. **Progressive hint:** Move behavior into `main(argv)` and use the `__main__` guard. **Verify:** Import the module while capturing output/files and assert no parser or write occurs; then assert `main([...])` performs the intended operation.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 9 — your work
# Short contract: Repair a module that parses arguments and writes files during import. Move behavior into `main(argv)` and use the `__main__` guard. Import the module while capturing output/file...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 10 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Define exit codes/messages for missing input, malformed data, existing output, and unexpected internal failure; decide which layers log. **Progressive hint:** Translate expected boundary failures once, without hiding tracebacks in tests. **Verify:** Exercise all four failure categories and assert their exit codes/messages; an injected unexpected error must remain visible in tests rather than being mislabeled.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 10 — your work
# Short contract: Define exit codes/messages for missing input, malformed data, existing output, and unexpected internal failure; decide which layers log. Translate expected boundary failures onc...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
